In [6]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  
# os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"  
from torch import nn


import numpy as np                                       
                           
import torch                                          

                              
from transformers import AutoModelForCausalLM, AutoTokenizer  
from tqdm import tqdm
import matplotlib.pyplot as plt  
import torch.nn.functional as F
import gc
import re
import copy

import sys
sys.path.append('..')
import JCBScope_utils
import JacobianScopes

# Move to GPU with optimal dtype
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = "cpu"


### Import pre-computed ranking

In [ ]:
mode = 'Temperature'
# mode = 'Semantic'
# mode = 'gradient_x_input'
# mode = 'Random'
# mode = 'IG'
# mode = 'Fisher'
mode = 'LOO'
presence_list = [0.2, 0.4, 0.6, 0.8, 1.0] 

# cutoff = 1000
cutoff = 5

In [8]:
import json

with open("../results/Llama-3.2-1B__LOO_KL_lambada_loo_results.json") as f:
    loo_results = json.load(f)

len(loo_results['results'])


JSONDecodeError: Expecting value: line 299 column 39 (char 6433)

In [ ]:
# Load the tokenizer and model

model_name = "meta-llama/Llama-3.2-1B"
model_name_short = model_name.split("/")[-1]
if device == "cpu":
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model = model.to(device)
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
    
embedding_layer = model.get_input_embeddings()
embed_device = embedding_layer.weight.device    

In [ ]:
front_pad = 0
back_pad = 0

front_strip = 0

# Get special tokens if available
bos_token_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id


In [ ]:
if mode in ('Semantic', 'IG', 'Fisher'):
    unnormalized_logits = True
else:
    unnormalized_logits = False
    

def get_influence_ranking(string, scope=None, fisher_method='low_rank', presence_list_ig=None):
    """Compute influence scores via JacobianScopes. scope from global mode if None; fisher_method when scope='fisher'; semantic_path_integral when scope='semantic'."""
    scope = scope or {'Temperature': 'temperature', 'Semantic': 'semantic', 'gradient_x_input': 'gradient_x_input',
                     'Random': 'random', 'IG': 'ig', 'Fisher': 'fisher'}.get(mode, 'temperature')
    presence_list_ig = presence_list_ig if presence_list_ig is not None else presence_list

    input_ids_list = tokenizer(string, add_special_tokens=False)["input_ids"]
    if eos_token_id is not None:
        input_ids_list += [eos_token_id] * back_pad

    decoded_tokens = tokenizer.batch_decode([[tid] for tid in input_ids_list], skip_special_tokens=True)
    grad_idx = [idx for idx in range(front_pad, len(decoded_tokens), 1)][front_strip:]
    tick_label_text = [decoded_tokens[idx] for idx in grad_idx]

    if scope == 'random':
        most_influential_local_idx = int(np.random.randint(0, len(grad_idx)))
        most_influential_idx = grad_idx[most_influential_local_idx]
        grad_vals = np.random.random(len(grad_idx)).astype(np.float32)
        ablated_indices = np.array([most_influential_local_idx])
        return most_influential_idx, grad_vals, ablated_indices, tick_label_text, grad_idx

    input_ids = torch.tensor([input_ids_list], dtype=torch.long).to(embed_device)
    attention_mask = torch.ones_like(input_ids, device=embed_device)
    seq_len = input_ids.size(1)

    d_model = embedding_layer.embedding_dim
    residual = nn.Parameter(torch.zeros(len(grad_idx), d_model, device=embed_device))
    presence = torch.ones(len(decoded_tokens), 1, device=embed_device)
    model.eval()
    forward_pass = JCBScope_utils.customize_forward_pass(
        model, residual, presence, input_ids, grad_idx, attention_mask
    )
    loss_position = seq_len - 2

    if scope == 'fisher':
        lm_head = JCBScope_utils.get_lm_head(model)
        grad_vals, _ = JacobianScopes.fisher_scope_scores(
            forward_pass, residual, loss_position, lm_head,method=fisher_method,
            k = 4
        )
    elif scope == 'temperature':
        grad_vals, _ = JacobianScopes.temperature_scope_scores(forward_pass, residual, loss_position)
        del _
    elif scope == 'semantic':
        grad_vals, _ = JacobianScopes.semantic_scope_scores(
            forward_pass, residual, loss_position,
            path_integral=False, grad_idx=grad_idx
        )
        del _
    elif scope == 'gradient_x_input':
        grad_vals, _ = JacobianScopes.gradient_x_input_scores(
            forward_pass, residual, loss_position, embedding_layer, input_ids, grad_idx
        )
        del _
    elif scope == 'ig':
        grad_vals, _ = JacobianScopes.semantic_scope_scores(
            forward_pass, residual, loss_position,
            grad_idx=grad_idx,
            path_integral=True,
            presence_ratios=presence_list ,
        )
        del _

    else:
        raise ValueError(f"Unknown scope: {scope!r}")
    if grad_vals.ndim > 1:
        grad_vals = grad_vals.squeeze()
    if not isinstance(grad_vals, np.ndarray):
        grad_vals = np.asarray(grad_vals, dtype=np.float32)

    most_influential_local_idx = int(np.argmax(grad_vals))
    most_influential_idx = grad_idx[most_influential_local_idx]
    ablated_indices = np.array([most_influential_local_idx])

    # del model
    gc.collect()
    if device != "cpu":
        torch.cuda.empty_cache()

    return most_influential_idx, grad_vals, ablated_indices, tick_label_text, grad_idx

In [5]:
import json
from pathlib import Path

# 1. Load each prompt from loo_results['results']
# 2. Use temperature/semantic scope to get most influential token index
# 3. Locate this index in ranked_token_indices (LOO ranking)
# 4. Save rank and ranking_pct into each result, with mode name in key

loo_json_path = Path("../results/Llama-3.2-1B__LOO_KL_lambada_loo_results.json")
with open(loo_json_path, "r", encoding="utf-8") as f:
    loo_results = json.load(f)

for i, item in enumerate(tqdm(loo_results["results"][:cutoff], desc="Processing prompts")):
    # if f"{mode}_rank" in item:
    #     continue
    prompt = item["prompt"]
    ranked_token_indices = item["ranked_token_indices"]
    print(prompt)
    
    most_influential_idx, grad_vals, ablated_indices, tick_label_text, grad_idx = get_influence_ranking(prompt)

    if most_influential_idx in ranked_token_indices:
        rank = ranked_token_indices.index(most_influential_idx)
        ranking_pct = (rank / len(ranked_token_indices)) * 100
    else:
        rank = None
        ranking_pct = None

    
    item[f"{mode}_rank"] = rank
    item[f"{mode}_ranking_pct"] = ranking_pct
    item[f"{mode}_influence_scores"] = grad_vals.tolist()
    print(f"[{i}] {mode}_rank = {rank}, {mode}_ranking_pct = {ranking_pct:.2g}" if ranking_pct is not None else f"[{i}] {mode}_rank = {rank}, {mode}_ranking_pct = None")
    most_influential_token_scope = tick_label_text[grad_idx[most_influential_idx]]
    print(f"Most influential token index (by {mode} scope): {most_influential_token_scope}")
    most_influential_token_loo = tick_label_text[ranked_token_indices[0]]
    print(f"Most influential token index (by LOO): {most_influential_token_loo}")

    del grad_vals, ablated_indices, tick_label_text, grad_idx, most_influential_idx, rank, ranking_pct
    if device != "cpu":
        torch.cuda.empty_cache()
    gc.collect()

    with open(loo_json_path, "w", encoding="utf-8") as f:
        json.dump(loo_results, f, indent=2, ensure_ascii=False)




JSONDecodeError: Expecting value: line 299 column 39 (char 6433)

In [4]:
results = loo_results["results"]
from math import sqrt

# Report ranking stats: where does the most influential token (by {mode} scope) rank in LOO?
rank_key, pct_key = f"{mode}_rank", f"{mode}_ranking_pct"
ranks = [r[rank_key] for r in results if r.get(rank_key) is not None]
pcts = [r[pct_key] for r in results if r.get(pct_key) is not None]
total = len(results)

if ranks:
    avg_rank = sum(ranks) / len(ranks)
    avg_pct = sum(pcts) / len(pcts) if pcts else float("nan") 

    # Compute SEM (standard error of the mean) for mean_ranking_pct if possible
    sem_ranking_pct = None
    if pcts and len(pcts) > 1:
        mean_pct = sum(pcts) / len(pcts)
        variance_pct = sum((x - mean_pct) ** 2 for x in pcts) / (len(pcts) - 1)
        sem_ranking_pct = sqrt(variance_pct / len(pcts))

    print(f"\n{mode} scope vs LOO ranking ({len(ranks)}/{total} prompts):")
    print(f"  Mean rank of most influential token in LOO: {avg_rank:.2f}")
    if sem_ranking_pct is not None:
        print(f"  Mean ranking percentile: {avg_pct:.2f} ± {sem_ranking_pct:.2f}%")
    else:
        print(f"  Mean ranking percentile: {avg_pct:.2f}%")
else:
    print("No rank data. Run the processing cell first.")

# Calculate how often most influential token's ranking percentile is within 5% of the top
within_5_pct_count = sum(1 for r in results if r.get(pct_key) is not None and r[pct_key] <= 5)
fraction_within_5_pct = within_5_pct_count / total if total else 0
print(f"\n{mode} scope most influential token is within top 5% of LOO ranking for {within_5_pct_count}/{total} prompts ({fraction_within_5_pct:.1%})")

# Save to master_results.json
label = f"Llama-3.2-1B__{mode}_lambada_loo_rank"
master_path = Path("../results/master_results.json")
master_path.parent.mkdir(parents=True, exist_ok=True)
master = {}
if master_path.exists():
    with open(master_path, "r", encoding="utf-8") as f:
        master = json.load(f)

entry = {
    "mean_rank": sum(ranks) / len(ranks) if ranks else None,
    "mean_ranking_pct": sum(pcts) / len(pcts) if pcts else None,
    "sem_mean_ranking_pct": sem_ranking_pct,
    "n_with_rank": len(ranks),
    "total": total,
    "within_top5_pct_count": within_5_pct_count,
    "fraction_within_top5_pct": fraction_within_5_pct,
}
master[label] = entry
with open(master_path, "w", encoding="utf-8") as f:
    json.dump(master, f, indent=2)
print(f"\nSaved to {master_path} (label={label})")



NameError: name 'loo_results' is not defined